In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import json
import random
import time
import subprocess
import io
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import xml.etree.ElementTree as ET

from PIL import Image
from tqdm.notebook import tqdm
from tqdm import tqdm as tqdm_std
from collections import Counter
from concurrent.futures import ThreadPoolExecutor, as_completed

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# Reprodutibilidade
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
torch.cuda.manual_seed_all(RANDOM_SEED)
torch.backends.cudnn.benchmark = True

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Dispositivo: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
BASE_DIR = '/content/drive/MyDrive/classification_exp'
OUTPUT_DIR = os.path.join(BASE_DIR, 'results')
MODEL_PATH = os.path.join(OUTPUT_DIR, 'best_model_stanford.pth')

os.makedirs(OUTPUT_DIR, exist_ok=True)

assert os.path.isdir(BASE_DIR), f'BASE_DIR nao encontrada: {BASE_DIR}'
assert os.path.isfile(MODEL_PATH), f'Modelo nao encontrado: {MODEL_PATH}'

print('BASE_DIR  :', BASE_DIR)
print('OUTPUT_DIR:', OUTPUT_DIR)
print('MODEL_PATH:', MODEL_PATH)

In [ ]:
# Versão sem treino:
# - não copia Images/ e Annotation/ completos
# - usa os arquivos direto do Drive nesta etapa
# - depois do split copia apenas o conjunto de teste

IMAGES_ROOT = os.path.join(BASE_DIR, 'Images')
ANNOT_ROOT = os.path.join(BASE_DIR, 'Annotation')

assert os.path.isdir(IMAGES_ROOT), f"Pasta de imagens nao encontrada: {IMAGES_ROOT}"
assert os.path.isdir(ANNOT_ROOT), f"Pasta de anotacoes nao encontrada: {ANNOT_ROOT}"

print('Modo sem treino: pulando copia completa para SSD.')
print('Apos o split, sera copiado apenas o conjunto de teste para disco local.')

breed_dirs = sorted([
    d for d in os.listdir(IMAGES_ROOT)
    if os.path.isdir(os.path.join(IMAGES_ROOT, d))
])

print(f'Total de pastas (racas): {len(breed_dirs)}')
print(f'Primeiras 5: {breed_dirs[:5]}')

In [ ]:
def parse_annotation(annot_path):
    try:
        xmin = ymin = xmax = ymax = None

        for _, elem in ET.iterparse(annot_path, events=("end",)):
            tag = elem.tag
            if tag == 'xmin':
                xmin = int(elem.text)
            elif tag == 'ymin':
                ymin = int(elem.text)
            elif tag == 'xmax':
                xmax = int(elem.text)
            elif tag == 'ymax':
                ymax = int(elem.text)
            elem.clear()

        if None not in (xmin, ymin, xmax, ymax):
            return (xmin, ymin, xmax, ymax)
    except Exception:
        return None

    return None


def process_one_image(img_full, annot_full, breed_name):
    bbox = parse_annotation(annot_full) if annot_full is not None else None
    return img_full, bbox, breed_name

In [ ]:
image_paths = []
bboxes = []
labels = []
breed_names_set = set()
bbox_found = 0

MAX_WORKERS = 32

print('Carregando imagens e anotacoes...')
futures = []

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
    for breed_dir in tqdm_std(breed_dirs, desc='Racas processadas'):
        breed_name = breed_dir.split('-', 1)[1] if '-' in breed_dir else breed_dir
        breed_name = breed_name.replace('_', ' ')
        breed_names_set.add(breed_name)

        breed_img_path = os.path.join(IMAGES_ROOT, breed_dir)
        breed_annot_path = os.path.join(ANNOT_ROOT, breed_dir)

        imgs = [
            f for f in os.listdir(breed_img_path)
            if f.lower().endswith(('.jpg', '.jpeg', '.png'))
        ]

        annot_set = set(os.listdir(breed_annot_path)) if os.path.isdir(breed_annot_path) else set()

        for img_file in imgs:
            img_full = os.path.join(breed_img_path, img_file)
            annot_name = os.path.splitext(img_file)[0]
            annot_full = os.path.join(breed_annot_path, annot_name) if annot_name in annot_set else None
            futures.append(ex.submit(process_one_image, img_full, annot_full, breed_name))

    for fut in tqdm_std(as_completed(futures), total=len(futures), desc='Parse XMLs'):
        img_full, bbox, breed_name = fut.result()
        image_paths.append(img_full)
        bboxes.append(bbox)
        labels.append(breed_name)
        if bbox is not None:
            bbox_found += 1

breed_names = sorted(list(breed_names_set))
label_to_idx = {name: idx for idx, name in enumerate(breed_names)}
idx_to_label = {idx: name for name, idx in label_to_idx.items()}
num_classes = len(breed_names)
labels_idx = [label_to_idx[l] for l in labels]

print(f'\nTotal de imagens: {len(image_paths)}')
print(f'Total de racas: {num_classes}')
print(f'Imagens com bounding box: {bbox_found} ({bbox_found/len(image_paths)*100:.1f}%)')

counts = Counter(labels)
min_breed = min(counts, key=counts.get)
max_breed = max(counts, key=counts.get)

print(f'Min amostras: {counts[min_breed]} ({min_breed})')
print(f'Max amostras: {counts[max_breed]} ({max_breed})')

In [ ]:
print('Realizando split estratificado...')
indices = list(range(len(image_paths)))

idx_train, idx_temp, _, _ = train_test_split(
    indices,
    labels_idx,
    test_size=0.30,
    stratify=labels_idx,
    random_state=RANDOM_SEED
)

temp_labels = [labels_idx[i] for i in idx_temp]

idx_val, idx_test, _, _ = train_test_split(
    idx_temp,
    temp_labels,
    test_size=0.50,
    stratify=temp_labels,
    random_state=RANDOM_SEED
)

X_train = [image_paths[i] for i in idx_train]
y_train = [labels_idx[i] for i in idx_train]
bb_train = [bboxes[i] for i in idx_train]

X_val = [image_paths[i] for i in idx_val]
y_val = [labels_idx[i] for i in idx_val]
bb_val = [bboxes[i] for i in idx_val]

X_test = [image_paths[i] for i in idx_test]
y_test = [labels_idx[i] for i in idx_test]
bb_test = [bboxes[i] for i in idx_test]

print(f'Treino:    {len(X_train)} imagens')
print(f'Validacao: {len(X_val)} imagens')
print(f'Teste:     {len(X_test)} imagens')

assert len(set(y_train)) == num_classes
assert len(set(y_val)) == num_classes
assert len(set(y_test)) == num_classes

print(f'Todas as {num_classes} classes presentes nos 3 conjuntos.')

In [ ]:
import shutil

LOCAL_DATA_TEST = '/content/local_data_test'
LOCAL_TEST_IMAGES = os.path.join(LOCAL_DATA_TEST, 'Images')
LOCAL_TEST_ANNOTS = os.path.join(LOCAL_DATA_TEST, 'Annotation')

os.makedirs(LOCAL_TEST_IMAGES, exist_ok=True)
os.makedirs(LOCAL_TEST_ANNOTS, exist_ok=True)

def copy_one_test_sample(img_src):
    breed_dir = os.path.basename(os.path.dirname(img_src))
    img_name = os.path.basename(img_src)
    img_stem = os.path.splitext(img_name)[0]

    src_annot_dir = os.path.join(ANNOT_ROOT, breed_dir)

    dst_img_dir = os.path.join(LOCAL_TEST_IMAGES, breed_dir)
    dst_annot_dir = os.path.join(LOCAL_TEST_ANNOTS, breed_dir)

    os.makedirs(dst_img_dir, exist_ok=True)
    os.makedirs(dst_annot_dir, exist_ok=True)

    dst_img = os.path.join(dst_img_dir, img_name)
    if not os.path.exists(dst_img):
        shutil.copy2(img_src, dst_img)

    src_annot = os.path.join(src_annot_dir, img_stem)
    dst_annot = os.path.join(dst_annot_dir, img_stem)

    annot_exists = False
    if os.path.exists(src_annot):
        if not os.path.exists(dst_annot):
            shutil.copy2(src_annot, dst_annot)
        annot_exists = True

    return dst_img, (dst_annot if annot_exists else None)

print('Copiando apenas o conjunto de teste para SSD local...')
t0 = time.time()

X_test_local = [None] * len(X_test)
bb_test_local = [None] * len(X_test)

max_workers = min(32, (os.cpu_count() or 8) * 2)

with ThreadPoolExecutor(max_workers=max_workers) as ex:
    futures = {ex.submit(copy_one_test_sample, img_src): idx for idx, img_src in enumerate(X_test)}

    for fut in tqdm(as_completed(futures), total=len(futures), desc='Copia teste'):
        idx = futures[fut]
        dst_img, dst_annot = fut.result()
        X_test_local[idx] = dst_img
        bb_test_local[idx] = bb_test[idx] if dst_annot is not None else None

print(f'Copia concluida em {time.time() - t0:.1f}s')
print(f'Total imagens de teste copiadas: {len(X_test_local)}')

In [ ]:
IMG_SIZE = 224

eval_transforms = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [ ]:
class DogBreedDataset(Dataset):
    def __init__(self, image_paths, labels, bboxes, transform=None, use_bbox=True):
        self.image_paths = image_paths
        self.labels = labels
        self.bboxes = bboxes
        self.transform = transform
        self.use_bbox = use_bbox

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        image = Image.open(self.image_paths[idx]).convert('RGB')
        label = self.labels[idx]

        if self.use_bbox and self.bboxes[idx] is not None:
            xmin, ymin, xmax, ymax = self.bboxes[idx]
            w, h = image.size

            margin_x = int((xmax - xmin) * 0.10)
            margin_y = int((ymax - ymin) * 0.10)

            xmin = max(0, xmin - margin_x)
            ymin = max(0, ymin - margin_y)
            xmax = min(w, xmax + margin_x)
            ymax = min(h, ymax + margin_y)

            image = image.crop((xmin, ymin, xmax, ymax))

        if self.transform:
            image = self.transform(image)

        return image, label

In [ ]:
TEST_BATCH_SIZE = 256

test_dataset = DogBreedDataset(
    X_test_local,
    y_test,
    bb_test_local,
    transform=eval_transforms,
    use_bbox=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=TEST_BATCH_SIZE,
    shuffle=False,
    num_workers=4,
    pin_memory=True,
    persistent_workers=True,
    prefetch_factor=2
)

print(f'Test batches: {len(test_loader)}')

In [ ]:
def create_model(num_classes):
    model = models.convnext_tiny(weights=models.ConvNeXt_Tiny_Weights.IMAGENET1K_V1)
    in_features = model.classifier[2].in_features
    model.classifier[2] = nn.Linear(in_features, num_classes)
    return model

model = create_model(num_classes).to(device)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f'Modelo carregado com {num_classes} classes.')
print(f'Parametros totais: {total_params:,}')
print(f'Parametros treinaveis: {trainable_params:,}')

In [ ]:
checkpoint = torch.load(MODEL_PATH, map_location=device)

if isinstance(checkpoint, dict) and 'state_dict' in checkpoint:
    state_dict = checkpoint['state_dict']
else:
    state_dict = checkpoint

clean_state_dict = {}
for k, v in state_dict.items():
    new_k = k
    if new_k.startswith('_orig_mod.'):
        new_k = new_k[len('_orig_mod.'):]
    if new_k.startswith('module.'):
        new_k = new_k[len('module.'):]
    clean_state_dict[new_k] = v

missing, unexpected = model.load_state_dict(clean_state_dict, strict=False)

print('Pesos carregados.')
print('Missing keys   :', missing)
print('Unexpected keys:', unexpected)

model.eval()

In [ ]:
all_preds = []
all_labels = []

print('Avaliando no conjunto de teste...')
with torch.no_grad():
    for images, labels_batch in tqdm(test_loader, desc='Teste'):
        images = images.to(device, non_blocking=True)

        if device.type == 'cuda':
            with torch.amp.autocast('cuda'):
                outputs = model(images)
        else:
            outputs = model(images)

        _, preds = torch.max(outputs, 1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels_batch.numpy())

all_preds = np.array(all_preds)
all_labels = np.array(all_labels)

test_accuracy = accuracy_score(all_labels, all_preds)
print(f'\nAcuracia Global no Teste: {test_accuracy:.4f} ({test_accuracy*100:.2f}%)')

In [ ]:
report = classification_report(
    all_labels,
    all_preds,
    labels=list(range(num_classes)),
    target_names=breed_names,
    zero_division=0,
    digits=3
)

print(report)

In [ ]:
breed_rows = []

for i, breed in enumerate(breed_names):
    mask = (all_labels == i)
    support = int(mask.sum())
    correct = int((all_preds[mask] == i).sum()) if support > 0 else 0
    errors = support - correct
    acc = (correct / support) if support > 0 else np.nan

    breed_rows.append({
        'breed_idx': i,
        'breed': breed,
        'support': support,
        'correct': correct,
        'errors': errors,
        'accuracy': acc
    })

breed_metrics_df = (
    pd.DataFrame(breed_rows)
    .sort_values(['accuracy', 'errors', 'support'], ascending=[True, False, False])
    .reset_index(drop=True)
)

top10_worst_df = breed_metrics_df.head(10).copy()
display(top10_worst_df)

In [ ]:
def misclassification_distribution_for_breed(breed_idx, all_labels, all_preds, breed_names):
    true_mask = (all_labels == breed_idx)
    mis_preds = all_preds[true_mask & (all_preds != breed_idx)]

    if len(mis_preds) == 0:
        return pd.DataFrame(columns=[
            'pred_idx', 'pred_breed', 'count', 'pct_errors', 'cum_pct_errors'
        ])

    counts = pd.Series(mis_preds).value_counts().sort_values(ascending=False)
    df = counts.rename_axis('pred_idx').reset_index(name='count')
    df['pred_breed'] = df['pred_idx'].map(lambda x: breed_names[int(x)])
    df['pct_errors'] = 100 * df['count'] / df['count'].sum()
    df['cum_pct_errors'] = df['pct_errors'].cumsum()

    return df[['pred_idx', 'pred_breed', 'count', 'pct_errors', 'cum_pct_errors']]


def top_until_threshold(dist_df, threshold=90.0):
    if dist_df.empty:
        return dist_df.copy()

    cutoff_idx = np.argmax(dist_df['cum_pct_errors'].values >= threshold)
    return dist_df.iloc[:cutoff_idx + 1].copy()

In [ ]:
for _, row in top10_worst_df.iterrows():
    breed_idx = int(row['breed_idx'])
    breed = row['breed']

    dist_df = misclassification_distribution_for_breed(
        breed_idx, all_labels, all_preds, breed_names
    )
    major_df = top_until_threshold(dist_df, threshold=90.0)

    print('=' * 100)
    print(f'Raça verdadeira: {breed}')
    print(f'Support: {row["support"]} | Acertos: {row["correct"]} | Erros: {row["errors"]} | Acurácia: {row["accuracy"]:.4f}')

    if dist_df.empty:
        print('Nenhum erro para essa raça.')
        continue

    top1 = dist_df.iloc[0]
    print(
        f'Principal confusão: {top1["pred_breed"]} '
        f'({int(top1["count"])} erros; {top1["pct_errors"]:.1f}% dos erros dessa raça)'
    )

    print('\nClasses que explicam ~90% dos erros:')
    display(
        major_df[['pred_breed', 'count', 'pct_errors', 'cum_pct_errors']]
        .rename(columns={
            'pred_breed': 'Classe predita',
            'count': 'Qtde',
            'pct_errors': '% dos erros',
            'cum_pct_errors': '% acumulado'
        })
        .style.format({
            '% dos erros': '{:.1f}',
            '% acumulado': '{:.1f}'
        })
    )

In [ ]:
for _, row in top10_worst_df.iterrows():
    breed_idx = int(row['breed_idx'])
    breed = row['breed']

    dist_df = misclassification_distribution_for_breed(
        breed_idx, all_labels, all_preds, breed_names
    )

    if dist_df.empty:
        continue

    major_df = top_until_threshold(dist_df, threshold=90.0)

    plt.figure(figsize=(10, max(4, 0.45 * len(major_df))))
    plt.barh(major_df['pred_breed'][::-1], major_df['pct_errors'][::-1])
    plt.xlabel('% dos erros da raça')
    plt.ylabel('Classe predita')
    plt.title(f'Distribuição dos erros — {breed}')
    plt.grid(axis='x', alpha=0.3)
    plt.tight_layout()
    plt.show()

In [ ]:
worst10_idx = top10_worst_df['breed_idx'].astype(int).tolist()
cm = confusion_matrix(all_labels, all_preds, labels=range(num_classes))

cm_worst10 = cm[worst10_idx, :]
row_sums = cm_worst10.sum(axis=1, keepdims=True)
cm_worst10_norm = np.divide(cm_worst10, row_sums, where=row_sums != 0)

plt.figure(figsize=(18, 6))
sns.heatmap(
    cm_worst10_norm,
    cmap='Reds',
    xticklabels=breed_names,
    yticklabels=[breed_names[i] for i in worst10_idx],
    cbar_kws={'label': 'Proporção por raça verdadeira'}
)
plt.xlabel('Classe predita')
plt.ylabel('Raça verdadeira (10 piores)')
plt.title('Matriz de confusão normalizada — foco nas 10 piores raças')
plt.xticks(rotation=90)
plt.tight_layout()
plt.show()

In [ ]:
analysis_dir = os.path.join(OUTPUT_DIR, 'analise_falhas_top10')
os.makedirs(analysis_dir, exist_ok=True)

top10_worst_df.to_csv(
    os.path.join(analysis_dir, 'top10_piores_racas.csv'),
    index=False,
    encoding='utf-8'
)

all_dist_rows = []
for _, row in top10_worst_df.iterrows():
    breed_idx = int(row['breed_idx'])
    breed = row['breed']

    dist_df = misclassification_distribution_for_breed(
        breed_idx, all_labels, all_preds, breed_names
    )

    if not dist_df.empty:
        dist_df.insert(0, 'true_breed_idx', breed_idx)
        dist_df.insert(1, 'true_breed', breed)
        all_dist_rows.append(dist_df)

if all_dist_rows:
    pd.concat(all_dist_rows, ignore_index=True).to_csv(
        os.path.join(analysis_dir, 'distribuicao_erros_top10.csv'),
        index=False,
        encoding='utf-8'
    )

print(f'Arquivos salvos em: {analysis_dir}')

In [ ]:
analysis_dir = os.path.join(OUTPUT_DIR, 'analise_falhas_top10')
os.makedirs(analysis_dir, exist_ok=True)

top10_worst_df.to_csv(
    os.path.join(analysis_dir, 'top10_piores_racas.csv'),
    index=False,
    encoding='utf-8'
)

all_dist_rows = []
for _, row in top10_worst_df.iterrows():
    breed_idx = int(row['breed_idx'])
    breed = row['breed']

    dist_df = misclassification_distribution_for_breed(
        breed_idx, all_labels, all_preds, breed_names
    )

    if not dist_df.empty:
        dist_df.insert(0, 'true_breed_idx', breed_idx)
        dist_df.insert(1, 'true_breed', breed)
        all_dist_rows.append(dist_df)

if all_dist_rows:
    pd.concat(all_dist_rows, ignore_index=True).to_csv(
        os.path.join(analysis_dir, 'distribuicao_erros_top10.csv'),
        index=False,
        encoding='utf-8'
    )

print(f'Arquivos salvos em: {analysis_dir}')

In [ ]:
print('=' * 70)
print('RESUMO DA ANALISE DE FALHAS')
print('=' * 70)
print(f'Acuracia global no teste: {test_accuracy:.4f} ({test_accuracy*100:.2f}%)')
print('\n10 piores raças por acurácia:')
display(top10_worst_df[['breed', 'support', 'correct', 'errors', 'accuracy']])

In [ ]:
error_rows = []

worst10_idx = set(top10_worst_df['breed_idx'].astype(int).tolist())

for i in range(len(all_labels)):
    true_idx = int(all_labels[i])
    pred_idx = int(all_preds[i])

    if true_idx in worst10_idx and true_idx != pred_idx:
        error_rows.append({
            'sample_idx': i,
            'image_path': X_test_local[i],
            'true_idx': true_idx,
            'true_breed': breed_names[true_idx],
            'pred_idx': pred_idx,
            'pred_breed': breed_names[pred_idx],
            'has_bbox': bb_test_local[i] is not None
        })

errors_top10_df = pd.DataFrame(error_rows).sort_values(
    ['true_breed', 'pred_breed']
).reset_index(drop=True)

print(f'Total de erros nas 10 piores raças: {len(errors_top10_df)}')
display(errors_top10_df.head(20))

In [ ]:
qualitative_dir = os.path.join(OUTPUT_DIR, 'analise_qualitativa_top10')
os.makedirs(qualitative_dir, exist_ok=True)

errors_csv_path = os.path.join(qualitative_dir, 'erros_top10_racas_qualitativo.csv')
errors_top10_df.to_csv(errors_csv_path, index=False, encoding='utf-8')

print(f'CSV salvo em: {errors_csv_path}')

In [ ]:
def load_image_with_optional_bbox(img_path, bbox=None, expand_bbox=True):
    image = Image.open(img_path).convert('RGB')

    if bbox is not None:
        xmin, ymin, xmax, ymax = bbox

        if expand_bbox:
            w, h = image.size
            margin_x = int((xmax - xmin) * 0.10)
            margin_y = int((ymax - ymin) * 0.10)
            xmin = max(0, xmin - margin_x)
            ymin = max(0, ymin - margin_y)
            xmax = min(w, xmax + margin_x)
            ymax = min(h, ymax + margin_y)

        image = image.crop((xmin, ymin, xmax, ymax))

    return image

In [ ]:
def plot_misclassified_examples_for_true_breed(
    true_breed_name,
    errors_df,
    sample_n=12,
    ncols=4,
    random_state=42
):
    subset = errors_df[errors_df['true_breed'] == true_breed_name].copy()

    if subset.empty:
        print(f'Nenhum erro encontrado para {true_breed_name}')
        return

    if len(subset) > sample_n:
        subset = subset.sample(sample_n, random_state=random_state)

    subset = subset.reset_index(drop=True)
    n = len(subset)
    nrows = int(np.ceil(n / ncols))

    plt.figure(figsize=(4 * ncols, 4 * nrows))

    for i, row in subset.iterrows():
        plt.subplot(nrows, ncols, i + 1)

        sample_idx = int(row['sample_idx'])
        bbox = bb_test_local[sample_idx] if row['has_bbox'] else None
        img = load_image_with_optional_bbox(row['image_path'], bbox=bbox, expand_bbox=True)

        plt.imshow(img)
        plt.axis('off')
        plt.title(
            f"Real: {row['true_breed']}\nPred: {row['pred_breed']}",
            fontsize=10
        )

    plt.suptitle(f'Erros qualitativos — raça verdadeira: {true_breed_name}', fontsize=14)
    plt.tight_layout()
    plt.show()

In [ ]:
for breed in top10_worst_df['breed'].tolist():
    plot_misclassified_examples_for_true_breed(
        true_breed_name=breed,
        errors_df=errors_top10_df,
        sample_n=12,
        ncols=4,
        random_state=42
    )

In [ ]:
def plot_top_confusion_examples_for_breed(
    true_breed_name,
    errors_df,
    sample_n=8,
    ncols=4,
    random_state=42
):
    subset = errors_df[errors_df['true_breed'] == true_breed_name].copy()

    if subset.empty:
        print(f'Nenhum erro encontrado para {true_breed_name}')
        return

    top_pred = subset['pred_breed'].value_counts().idxmax()
    subset = subset[subset['pred_breed'] == top_pred].copy()

    if len(subset) > sample_n:
        subset = subset.sample(sample_n, random_state=random_state)

    subset = subset.reset_index(drop=True)
    n = len(subset)
    nrows = int(np.ceil(n / ncols))

    plt.figure(figsize=(4 * ncols, 4 * nrows))

    for i, row in subset.iterrows():
        plt.subplot(nrows, ncols, i + 1)

        sample_idx = int(row['sample_idx'])
        bbox = bb_test_local[sample_idx] if row['has_bbox'] else None
        img = load_image_with_optional_bbox(row['image_path'], bbox=bbox, expand_bbox=True)

        plt.imshow(img)
        plt.axis('off')
        plt.title(
            f"Real: {row['true_breed']}\nPred: {row['pred_breed']}",
            fontsize=10
        )

    plt.suptitle(
        f'Principal confusão — {true_breed_name} confundido com {top_pred}',
        fontsize=14
    )
    plt.tight_layout()
    plt.show()

In [ ]:
qualitative_images_dir = os.path.join(qualitative_dir, 'imagens_erros')
os.makedirs(qualitative_images_dir, exist_ok=True)

copied = 0

for _, row in tqdm(errors_top10_df.iterrows(), total=len(errors_top10_df), desc='Salvando imagens de erro'):
    true_breed = row['true_breed'].replace('/', '_')
    pred_breed = row['pred_breed'].replace('/', '_')

    dst_dir = os.path.join(
        qualitative_images_dir,
        true_breed,
        f'pred_{pred_breed}'
    )
    os.makedirs(dst_dir, exist_ok=True)

    src = row['image_path']
    dst = os.path.join(dst_dir, os.path.basename(src))

    if not os.path.exists(dst):
        shutil.copy2(src, dst)
        copied += 1

print(f'Imagens copiadas: {copied}')
print(f'Pasta raiz: {qualitative_images_dir}')

In [ ]:
qual_summary = (
    errors_top10_df.groupby(['true_breed', 'pred_breed'])
    .size()
    .reset_index(name='count')
    .sort_values(['true_breed', 'count'], ascending=[True, False])
)

display(qual_summary)